# Baseline3 Adding Data Image Captions (model-reasoning based)

- after finishing the BOTH can dot this work.
- idea: 模型改进，这里的模型用的是很老的 20年 meta 的模型 "fairseq" => 拿它的生成器 "generator"

## Package Loading

In [51]:
# 添加OFA目录到Python路径，以便导入utils、tasks、models模块
import sys
import os

# 获取项目根目录（notebook在data_preprocess/baseline/目录下）
project_root = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
ofa_dir = os.path.join(project_root, 'OFA')

# 将OFA目录添加到Python路径
if os.path.exists(ofa_dir) and ofa_dir not in sys.path:
    sys.path.insert(0, ofa_dir)
    print(f"✓ 已添加OFA目录到Python路径: {ofa_dir}")
elif not os.path.exists(ofa_dir):
    print(f"⚠ 警告: OFA目录不存在: {ofa_dir}")
    print("  请确保已下载OFA代码到项目根目录下的OFA文件夹")
else:
    print(f"✓ OFA目录已在Python路径中: {ofa_dir}")



✓ OFA目录已在Python路径中: c:\Users\Aiur\Hateful-Image-Project\OFA


In [52]:
# 安装OFA需要的额外依赖
! pip install timm "numpy<2" opencv-python ftfy==6.0.3 editdistance "omegaconf>=2.0.5,<2.1"

  Using cached omegaconf-2.0.6-py3-none-any.whl.metadata (3.0 kB)
  Using cached omegaconf-2.0.5-py3-none-any.whl.metadata (3.0 kB)


Requested omegaconf<2.1,>=2.0.5 from https://files.pythonhosted.org/packages/d0/eb/9d63ce09dd8aa85767c65668d5414958ea29648a0eec80a4a7d311ec2684/omegaconf-2.0.6-py3-none-any.whl has invalid metadata: .* suffix can only be used with `==` or `!=` operators
    PyYAML (>=5.1.*)
            ~~~~~~^
Please use pip<24.1 if you need to use this version.
Requested omegaconf<2.1,>=2.0.5 from https://files.pythonhosted.org/packages/e5/f6/043b6d255dd6fbf2025110cea35b87f4c5100a181681d8eab496269f0d5b/omegaconf-2.0.5-py3-none-any.whl has invalid metadata: .* suffix can only be used with `==` or `!=` operators
    PyYAML (>=5.1.*)
            ~~~~~~^
Please use pip<24.1 if you need to use this version.
ERROR: Ignored the following yanked versions: 1.0.0, 1.0.1, 1.0.2, 2.0.0rc1, 2.0.0rc2, 2.0.0rc22, 2.0.0rc23, 2.0.0rc24, 2.0.0rc25, 2.0.0rc26, 2.0.0rc27, 2.0.0rc28, 2.0.0rc29, 2.0.1rc1, 2.0.1rc2, 2.0.1rc3, 2.0.1rc4, 2.0.1rc5, 2.2.0
ERROR: Could not find a version that satisfies the requirement omegaconf<

In [53]:
# 修复依赖冲突：使用兼容的版本
# omegaconf 2.1.1 修复了元数据问题，pytorch_lightning 使用兼容 torch 1.12 的版本
! pip install "omegaconf==2.1.1" "pytorch_lightning==1.9.5" "torch==1.12.0"

In [54]:
# ! pip install cython hydra-core omegaconf sacrebleu
# ! pip install git+https://github.com/pytorch/fairseq.git

In [55]:
# 安装额外依赖：evaluate 库替代了 datasets 中的 load_metric
! pip install evaluate einops

In [56]:
import os
import sys
import torch
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# 确保OFA目录在路径中
project_root = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
ofa_dir = os.path.join(project_root, 'OFA')
if ofa_dir not in sys.path:
    sys.path.insert(0, ofa_dir)

# 导入fairseq模块
from fairseq import utils, tasks
from fairseq import checkpoint_utils

# 导入OFA模块 (使用绝对导入)
import sys
sys.path.insert(0, ofa_dir) if ofa_dir not in sys.path else None

from utils.eval_utils import eval_step
from tasks.mm_tasks.caption import CaptionTask
from models.ofa import OFAModel
from PIL import Image

from tqdm.auto import tqdm

# Register caption task
tasks.register_task('caption', CaptionTask)

# turn on cuda if GPU is available
use_cuda = torch.cuda.is_available()
# use fp16 only when GPU is available
use_fp16 = False

- 检测计算资源

In [57]:
# ! pip install psutil torch

In [58]:
import psutil
import torch

def get_system_resources():
    # 获取CPU信息
    cpu_count = psutil.cpu_count(logical=False)  # 物理CPU核心数
    cpu_freq = psutil.cpu_freq()  # CPU频率
    memory = psutil.virtual_memory()  # 内存使用情况
    swap = psutil.swap_memory()  # 交换内存使用情况

    # 获取GPU信息（仅适用于安装了PyTorch的情况下）
    gpus = []
    if torch.cuda.is_available():
        for i in range(torch.cuda.device_count()):
            gpu_info = {}
            
            # 获取 GPU 名称
            gpu_info['GPU'] = torch.cuda.get_device_name(i)

            # 获取总内存
            try:
                gpu_info['Total Memory (MB)'] = torch.cuda.get_device_properties(i).total_memory / 1024**2
            except Exception as e:
                gpu_info['Total Memory (MB)'] = None
            
            # 获取分配的内存
            try:
                gpu_info['Memory Allocated (MB)'] = torch.cuda.memory_allocated(i) / 1024**2
            except Exception as e:
                gpu_info['Memory Allocated (MB)'] = None
            
            # 获取缓存的内存
            try:
                gpu_info['Memory Cached (MB)'] = torch.cuda.memory_reserved(i) / 1024**2
            except Exception as e:
                gpu_info['Memory Cached (MB)'] = None
            
            gpus.append(gpu_info)
    else:
        gpus.append({'GPU': 'No GPU available'})

    # 打印系统资源信息
    print("CPU Info:")
    print(f"  Physical cores: {cpu_count}")
    print(f"  Max Frequency: {cpu_freq.max} MHz")
    print(f"  Current Frequency: {cpu_freq.current} MHz")
    print(f"  Memory Total: {memory.total / 1024**3:.2f} GB")
    print(f"  Memory Available: {memory.available / 1024**3:.2f} GB")
    print(f"  Swap Total: {swap.total / 1024**3:.2f} GB")
    print(f"  Swap Used: {swap.used / 1024**3:.2f} GB")

    print("\nGPU Info:")
    for gpu in gpus:
        print(f"  GPU: {gpu.get('GPU', 'N/A')}")
        
        total_memory = gpu.get('Total Memory (MB)', 'N/A')
        print(f"    Total Memory: {total_memory if total_memory == 'N/A' else f'{total_memory:.2f} MB'}")
        
        memory_allocated = gpu.get('Memory Allocated (MB)', 'N/A')
        print(f"    Memory Allocated: {memory_allocated if memory_allocated == 'N/A' else f'{memory_allocated:.2f} MB'}")
        
        memory_cached = gpu.get('Memory Cached (MB)', 'N/A')
        print(f"    Memory Cached: {memory_cached if memory_cached == 'N/A' else f'{memory_cached:.2f} MB'}")

# 调用函数
get_system_resources()

CPU Info:
  Physical cores: 24
  Max Frequency: 2200.0 MHz
  Current Frequency: 2200.0 MHz
  Memory Total: 31.63 GB
  Memory Available: 12.02 GB
  Swap Total: 4.44 GB
  Swap Used: 0.96 GB

GPU Info:
  GPU: No GPU available
    Total Memory: N/A
    Memory Allocated: N/A
    Memory Cached: N/A


## Model Loading and Implement

- 超参数调配: 
> overrides={"bpe_dir":"utils/BPE", "eval_cider":False, "beam":5, "max_len_b":16, "no_repeat_ngram_size":3, "seed":7}
- 模型加载：
    - load => load model => ensemble task

- models (本方法应该融合了多种模型)
- 改LLM模型，这里的模型太老了...

### 下载 Caption Finetuned 模型

**重要**: `ofa_base.pt` 是预训练模型,使用 `denoising_unify` 任务。要生成图像字幕,需要下载已经在 Caption 任务上 finetuned 的模型。

In [59]:
# 下载 Caption finetuned 模型
import urllib.request
import os

# 模型下载URL和保存路径
model_url = "https://ofa-beijing.oss-cn-beijing.aliyuncs.com/checkpoints/caption_base_best.pt"
save_dir = os.path.join(ofa_dir, "checkpoints")
save_path = os.path.join(save_dir, "caption_base_best.pt")

# 确保目录存在
os.makedirs(save_dir, exist_ok=True)

# 检查是否已下载
if os.path.exists(save_path):
    file_size = os.path.getsize(save_path) / (1024 * 1024)  # MB
    print(f"✓ 模型已存在: {save_path}")
    print(f"  文件大小: {file_size:.1f} MB")
else:
    print(f"正在下载 Caption finetuned 模型...")
    print(f"  URL: {model_url}")
    print(f"  保存到: {save_path}")
    print(f"  预计大小: ~700 MB, 可能需要几分钟...")
    
    try:
        # 下载带进度显示
        def show_progress(block_num, block_size, total_size):
            downloaded = block_num * block_size
            percent = min(downloaded * 100.0 / total_size, 100)
            mb_downloaded = downloaded / (1024 * 1024)
            mb_total = total_size / (1024 * 1024)
            print(f"\r  进度: {percent:.1f}% ({mb_downloaded:.1f}/{mb_total:.1f} MB)", end='')
        
        urllib.request.urlretrieve(model_url, save_path, reporthook=show_progress)
        print("\n✓ 下载完成!")
    except Exception as e:
        print(f"\n✗ 下载失败: {e}")
        print("\n请手动下载:")
        print(f"  1. 访问: {model_url}")
        print(f"  2. 保存到: {save_path}")

✓ 模型已存在: c:\Users\Aiur\Hateful-Image-Project\OFA\checkpoints\caption_base_best.pt
  文件大小: 2149.8 MB


In [60]:
# 模型加载配置 - 使用 Caption finetuned 模型
import os
import torch

# 优先使用 caption finetuned 模型
checkpoint_paths = [
    os.path.join(ofa_dir, 'checkpoints/caption_base_best.pt'),  # 首选: Caption finetuned
    'checkpoints/caption_base_best.pt',
    os.path.join(ofa_dir, 'checkpoints/ofa_base.pt'),  # 备选: 预训练模型(不推荐)
    'checkpoints/ofa_base.pt',
]

available_checkpoint = None
for path in checkpoint_paths:
    if os.path.exists(path):
        available_checkpoint = path
        print(f"✓ 找到模型: {path}")
        if 'caption' in path:
            print("  ✓ 使用 Caption finetuned 模型 (推荐)")
        else:
            print("  ⚠ 使用预训练模型 (可能不兼容,建议下载 caption_base_best.pt)")
        break

if not available_checkpoint:
    raise FileNotFoundError(
        "未找到模型文件!\n"
        "请运行上面的单元格下载 caption_base_best.pt"
    )

print("正在加载模型...")

# BPE 目录路径 (使用 OFA 目录下的完整路径)
bpe_dir = os.path.join(ofa_dir, "utils", "BPE")
print(f"  BPE 目录: {bpe_dir}")

# 使用标准方法加载,禁用 CIDEr 评估以避免加载缓存文件
models, cfg, task = checkpoint_utils.load_model_ensemble_and_task(
    utils.split_paths(available_checkpoint),
    arg_overrides={
        "bpe_dir": bpe_dir,
        "eval_cider": False,  # 禁用 CIDEr 评估
        "eval_args": '{}',
        "eval_print_samples": False,
    }
)
print("✓ 模型成功加载")

# 模型评估模式并移到GPU
for model in models:
    model.eval()
    if use_fp16:
        model.half()
    if use_cuda and not cfg.distributed_training.pipeline_model_parallel:
        model.cuda()
    model.prepare_for_inference_(cfg)

# 初始化生成器
generator = task.build_generator(models, cfg.generation)
print("✓ 模型加载完成")
print(f"  任务类型: {cfg.task._name}")
print(f"  Beam size: {cfg.generation.beam}")
print(f"  Max length: {cfg.generation.max_len_b}")

✓ 找到模型: c:\Users\Aiur\Hateful-Image-Project\OFA\checkpoints/caption_base_best.pt
  ✓ 使用 Caption finetuned 模型 (推荐)
正在加载模型...
  BPE 目录: c:\Users\Aiur\Hateful-Image-Project\OFA\utils\BPE


2025-11-17 15:27:27 | INFO | tasks.ofa_task | source dictionary: 59457 types
2025-11-17 15:27:27 | INFO | tasks.ofa_task | target dictionary: 59457 types
2025-11-17 15:27:27 | INFO | tasks.ofa_task | target dictionary: 59457 types


✓ 模型成功加载
✓ 模型加载完成
  任务类型: caption
  Beam size: 5
  Max length: 200


## Preprocessing

- 图像处理
- txt prompt 并没有进行优化引导
    - idea: 可以进行 prompt 优化

In [61]:
# Image transform
from torchvision import transforms
mean = [0.5, 0.5, 0.5]
std = [0.5, 0.5, 0.5]

patch_resize_transform = transforms.Compose([
    lambda image: image.convert("RGB"),
    transforms.Resize((cfg.task.patch_image_size, cfg.task.patch_image_size), interpolation=Image.BICUBIC),
    transforms.ToTensor(),
    transforms.Normalize(mean=mean, std=std),
])

# Text preprocess
bos_item = torch.LongTensor([task.src_dict.bos()])
eos_item = torch.LongTensor([task.src_dict.eos()])
pad_idx = task.src_dict.pad()
def encode_text(text, length=None, append_bos=False, append_eos=False):
    s = task.tgt_dict.encode_line(
        line=task.bpe.encode(text),
        add_if_not_exist=False,
        append_eos=False
    ).long()
    if length is not None:
        s = s[:length]
    if append_bos:
        s = torch.cat([bos_item, s])
    if append_eos:
        s = torch.cat([s, eos_item])
    return s

# Construct input for caption task
# 直接读图，处理图
def construct_sample(image: Image):
    patch_image = patch_resize_transform(image).unsqueeze(0)
    patch_mask = torch.tensor([True])
    src_text = encode_text(" what does the image describe?", append_bos=True, append_eos=True).unsqueeze(0)
    src_length = torch.LongTensor([s.ne(pad_idx).long().sum() for s in src_text])
    sample = {
        "id":np.array(['42']),
        "net_input": {
            "src_tokens": src_text,
            "src_lengths": src_length,
            "patch_images": patch_image,
            "patch_masks": patch_mask
        }
    }
    return sample
  
# Function to turn FP32 to FP16
def apply_half(t):
    if t.dtype is torch.float32:
        return t.to(dtype=torch.half)
    return t

c:\Users\Aiur\miniconda3\envs\hateful-image-ofa\lib\site-packages\torchvision\transforms\transforms.py:332: UserWarning: Argument 'interpolation' of type int is deprecated since 0.13 and will be removed in 0.15. Please use InterpolationMode enum.
  warnings.warn(


## Run Reference and Reasoning

In [62]:
# 加载数据集信息
# Notebook 位置: data_preprocess/baseline/
# 数据文件位置: data/all_data/info_fine_grained.csv
info_fp = "../../data/all_data/info_fine_grained.csv"
info_df = pd.read_csv(info_fp)
print(f"✓ 加载了 {len(info_df)} 条数据")
info_df.head()

✓ 加载了 10000 条数据


,id,img,label,text,split,text_idx,pseudo_text_idx,pseudo_img_idx,gold_pc,gold_attack,...,religion_pc,sex_pc,attack_empty_attack,contempt_attack,dehumanizing_attack,exclusion_attack,inciting_violence_attack,inferiority_attack,mocking_attack,slurs_attack
0,42953,img/42953.png,0.0,its their character not their color that matters,train,0,2161,0,['pc_empty'],['attack_empty'],...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,23058,img/23058.png,0.0,don't be afraid to love again everyone is not ...,train,1,2931,2816,['pc_empty'],['attack_empty'],...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,13894,img/13894.png,0.0,putting bows on your pet,train,2,0,5188,['pc_empty'],['attack_empty'],...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,37408,img/37408.png,0.0,i love everything and everybody! except for sq...,train,3,3542,3435,['pc_empty'],['attack_empty'],...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,82403,img/82403.png,0.0,"everybody loves chocolate chip cookies, even h...",train,4,4614,1,['pc_empty'],['attack_empty'],...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


## Question: 哪来的 hateful_memes_masked?
- 一个存储图像的文件夹：(但为啥有 masked?)

In [63]:
# 路径修改
img_folder = "../../data/hateful_memes_masked"

captions = []
for img_fn in tqdm(info_df['img'].str.split('/').str[1]):
    img_fp = os.path.join(img_folder, img_fn)

    img = Image.open(img_fp)

    # Construct input sample & preprocess for GPU if cuda available
    sample = construct_sample(img)
    sample = utils.move_to_cuda(sample) if use_cuda else sample
    sample = utils.apply_to_sample(apply_half, sample) if use_fp16 else sample

    # Run eval step for caption
    with torch.no_grad():
        result, scores = eval_step(task, generator, models, sample)

    captions.append(result[0]['caption'])

    plt.imshow(img)
    plt.title(result[0]['caption'])
    plt.show()

  0%|          | 0/10000 [00:00<?, ?it/s]

FileNotFoundError: [Errno 2] No such file or directory: '../../data/hateful_memes_masked\\42953.png'

In [ ]:
info_df['caption'] = captions
float_cols = info_df.select_dtypes(float).columns
info_df[float_cols] = info_df.select_dtypes(float).astype('Int64')

In [ ]:
info_df.to_csv("../../data/hateful_memes/hateful_memes_expanded.csv")